# Uploading Content

Upload video, audio, or image content to the platform as assets. This notebook covers both direct (local file) and URL-based upload methods.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import os
import time

from twelvelabs import TwelveLabs

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")

client = TwelveLabs(api_key=API_KEY)

## When You Need This

Before you can build a knowledge store, you need content. This guide covers both upload methods:

- **Direct upload** — local video and audio files up to 200 MB, local images up to 32 MB
- **URL upload** — public video and audio URLs up to 4 GB, public image URLs up to 32 MB

## Helper: Wait for Asset Processing

Assets are processed asynchronously after upload. This helper polls until the asset reaches `ready` or `failed` status.

In [ ]:
def wait_for_asset_ready(
    asset_id: str,
    interval: int = 5,
    timeout: int = 600,
):
    """Poll an asset until it reaches 'ready' or 'failed' status.

    Args:
        asset_id: The identifier of the asset to monitor.
        interval: Seconds between polling attempts.
        timeout: Maximum seconds to wait before raising an error.

    Returns:
        The Asset object once it reaches 'ready' status.

    Raises:
        Exception: If the asset fails processing or the timeout is exceeded.
    """
    elapsed = 0
    while elapsed < timeout:
        asset = client.assets.retrieve(asset_id=asset_id)

        if asset.status == "ready":
            print(f"Asset {asset_id} is ready.")
            return asset
        elif asset.status == "failed":
            raise Exception(f"Asset {asset_id} processing failed.")

        print(f"Asset status: {asset.status} (elapsed: {elapsed}s)")
        time.sleep(interval)
        elapsed += interval

    raise Exception(f"Timeout after {timeout}s waiting for asset {asset_id}.")

## Direct Upload (Local File)

Best for local video and audio files up to 200 MB, or local images up to 32 MB. Pass the file opened in binary read mode; the platform detects the kind of content automatically.

In [ ]:
# Direct upload — local file
LOCAL_FILE_PATH = "video.mp4"  # Replace with your file path

asset = client.assets.create(method="direct", file=open(LOCAL_FILE_PATH, "rb"))
print(f"Asset ID: {asset.id}, Status: {asset.status}, File type: {asset.file_type}")

## URL Upload (Remote File)

Best for files already hosted online. Public video and audio URLs support up to 4 GB; public image URLs support up to 32 MB. The URL must be publicly accessible — the platform fetches the file directly.

In [ ]:
# URL upload — remote file
REMOTE_VIDEO_URL = "https://example.com/large-video.mp4"  # Replace with your URL

asset = client.assets.create(method="url", url=REMOTE_VIDEO_URL)
print(f"Asset ID: {asset.id}, Status: {asset.status}, File type: {asset.file_type}")

## Waiting for Processing

Assets are processed asynchronously. You must wait for the asset to reach `ready` status before adding it to a knowledge store.

In [ ]:
# Wait for the uploaded asset to be ready
asset_id = asset.id
ready_asset = wait_for_asset_ready(asset_id)
print(f"Status: {ready_asset.status}, File type: {ready_asset.file_type}")

## Optional Features: HLS Streaming and Thumbnails

Enable HLS streaming and thumbnail generation at upload time. Both options apply to video and audio assets.

In [ ]:
# Upload with HLS streaming and thumbnail generation enabled
asset_with_extras = client.assets.create(
    method="url",
    url="https://example.com/video.mp4",
    enable_hls=True,
    enable_thumbnail=True,
)
print(f"Asset ID: {asset_with_extras.id}, Status: {asset_with_extras.status}")

## Common Pitfalls

- **File too large for direct upload** — Local video and audio files are capped at 200 MB, and local images at 32 MB. Use the URL method for larger video and audio files (up to 4 GB).
- **URL not accessible** — The platform must be able to fetch the URL. Check that it is publicly reachable.
- **Forgetting to wait** — An asset must be `ready` before you can add it to a knowledge store. Always poll for status.

## Next Steps

- [Building Knowledge Stores](building_knowledge_stores.ipynb) — organize uploaded assets into queryable collections
- [Ingestion Config](ingestion_config.ipynb) — control what Jockey extracts from your videos and images
- [Querying](querying.ipynb) — ask questions about your videos and images

**API Reference:** [POST /assets](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/upload-content/direct-uploads/create)